In [ ]:
import xarray as xr
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
import cartopy.io.shapereader as shpreader
import cartopy
from utils import dataloader, preprocessing

In [ ]:
#make a barplots dir in figures
if not os.path.exists('figures/rpss'):
    os.makedirs('figures/rpss')

get y test for masking degenerate gridpoints

In [ ]:
download = True

In [ ]:
obs = "IMD"
model = "GEFS"
domain = [67, 98, 7, 38] # west east south north. for Unet's check that lat and lot make a square divisible by 8, ie 24x24, 32x32, 64x64
season = "May-Sep"
years = (1989, 2018)
week = "wk3-4"
xGEFS, yGEFS = dataloader.get_data(years=years, download = download,week=week,obs=obs, domain=domain, season=season,
                           model=model,regrid=None)



labeler_train = preprocessing.rolling_labeler(yGEFS.fillna(0)
                                                  ,window=1) 
yGEFS_labeled = labeler_train(yGEFS) # get y test for masking degenerate gridpoints

def count_unique(values):
    return len(np.unique(values))
unique_counts = xr.apply_ufunc(count_unique, yGEFS_labeled, input_core_dims=[['T']], vectorize=True)
# Mask grid points with less than 3 unique labels or NaNs
mask1 = (unique_counts <3)
mask2 = np.isnan(yGEFS_labeled).any(dim='T')
#combine masks
mask_GEFS = mask1 | mask2

In [ ]:
obs = "IMD" 
model = "IITM"
domain = [67, 98.5, 7, 38.5]  # West East South North, for Unet's check that lat and lot make a square divisible by 8, ie 24x24, 32x32, 64x64
season = "May-Sep"
years = (2003,2018)
week = "wk3-4"
xIITM, yIITM = dataloader.get_data(years=years, download = download,week=week,obs=obs, domain=domain, season=season,
                           model=model,regrid=None)



labeler_train = preprocessing.rolling_labeler(yIITM.fillna(0)
                                                  ,window=1) 
yIITM_labeled = labeler_train(yIITM) # get y test for masking degenerate gridpoints

def count_unique(values):
    return len(np.unique(values))
unique_counts = xr.apply_ufunc(count_unique, yIITM_labeled, input_core_dims=[['T']], vectorize=True)
# Mask grid points with less than 3 unique labels or NaNs
mask1 = (unique_counts <3)
mask2 = np.isnan(yIITM).any(dim='T')
#combine masks
mask_IITM = mask1 | mask2

In [ ]:
obs = "IMD"
model = "ECMWF"
domain = [66, 100, 7, 39] # west east south north. for Unet's check that lat and lot make a square divisible by 8, ie 24x24, 32x32, 64x64
season = "May-Sep"
years = (2003, 2022) #ECMWF hindcast range 20 years, behind the forecats year. Here we want a forecast year of 2023 so we use 2003-2022

xECMWF, yECMWF = dataloader.get_data(years=years, download = download,week=week,obs=obs, domain=domain, season=season,
                           model=model,regrid=None)



labeler_train = preprocessing.rolling_labeler(yECMWF.fillna(0)
                                                  ,window=1) 
yECMWF_labeled = labeler_train(yECMWF) # get y test for masking degenerate gridpoints

def count_unique(values):
    return len(np.unique(values))
unique_counts = xr.apply_ufunc(count_unique, yECMWF_labeled, input_core_dims=[['T']], vectorize=True)
# Mask grid points with less than 3 unique labels or NaNs
mask1 = (unique_counts <3)
mask2 = np.isnan(yECMWF).any(dim='T')
#combine masks
mask_ECMWF = mask1 | mask2

In [ ]:
obs = "IMD"
model = 'GEFS'
domain = [67, 98, 7, 38] # west east south north. for Unet's check that lat and lot make a square divisible by 8, ie 24x24, 32x32, 64x64
season = "May-Sep"
n_bootstraps = 10
years = (2003, 2018)

xmme, ymme = dataloader.get_data(years=years, download = download,week=week,obs=obs, domain=domain, season=season,
                           model=model,regrid=1)



labeler_train = preprocessing.rolling_labeler(ymme.fillna(0)
                                                  ,window=1) 
ymme_labeled = labeler_train(ymme) # get y test for masking degenerate gridpoints

def count_unique(values):
    return len(np.unique(values))
unique_counts = xr.apply_ufunc(count_unique, ymme_labeled, input_core_dims=[['T']], vectorize=True)
# Mask grid points with less than 3 unique labels or NaNs
mask1 = (unique_counts <3)
mask2 = np.isnan(ymme).any(dim='T')
#combine masks
mask_MME = mask1 | mask2

In [ ]:
import os
import xarray as xr

# --------------------------------------------------
# Settings
# --------------------------------------------------
base_dir = "outputs"
lead_file = "rpss_test_wk3-4.nc"
architectures = ["unet", "ELR"]

# Models by period
models_full = ["IITM_IMD", "GEFS_IMD", "ECMWF_IMD"]
models_common = ["IITM_IMD", "GEFS_IMD", "ECMWF_IMD", "MME_IMD", "2MME_IMD"]
models_mme = ["MME_IMD"]
models_2mme = ["2MME_IMD"]

period_to_models = {
    "Full period": models_full,
    "Common Period": models_common,
    "MME": models_mme,
    "2MME": models_2mme,
}

# --------------------------------------------------
# Load all RPSS maps into nested dict
# --------------------------------------------------
all_rpss = {arch: {} for arch in architectures}

for arch in architectures:
    for period, models in period_to_models.items():
        all_rpss[arch][period] = {}
        for model in models:
            model_dir = os.path.join(base_dir, period, model)
            file_path = os.path.join(model_dir, f"{arch}_{lead_file}")

            if not os.path.exists(file_path):
                print(f"Missing file: {file_path}")
                continue

            ds = xr.open_dataset(file_path)
            if "__xarray_dataarray_variable__" in ds:
                ds = ds.rename_vars({"__xarray_dataarray_variable__": "rpss"})
            rpss = ds.rpss

            # Mean over bootstrap if available
            if "bootstrap" in rpss.dims:
                rpss = rpss.mean("bootstrap")

            # Store with clean model name
            model_name = model.replace("_IMD", "")
            all_rpss[arch][period][model_name] = rpss


#load raw RPSS maps 

rpss_ecwmf_raw = xr.open_dataarray('outputs/Common Period/ECMWF_IMD/RAW_rpss_test_wk3-4.nc')
rpss_gefs_raw = xr.open_dataarray('outputs/Common Period/GEFS_IMD/RAW_rpss_test_wk3-4.nc')
rpss_iitm_raw = xr.open_dataarray('outputs/Common Period/IITM_IMD/RAW_rpss_test_wk3-4.nc')

#add to all_rpss dict
all_rpss['raw'] = {}
all_rpss['raw']['Common Period'] = {
    'ECMWF': rpss_ecwmf_raw,
    'GEFS': rpss_gefs_raw,
    'IITM': rpss_iitm_raw
}


In [ ]:
# Harmonize UNet masks to ELR
for period in all_rpss.get("ELR", {}):
    for model in all_rpss["ELR"][period]:
        if model not in all_rpss.get("unet", {}).get(period, {}):
            continue

        elr_da = all_rpss["ELR"][period][model]
        unet_da = all_rpss["unet"][period][model]

        # Select mask depending on period and model
        if period == "Full period":
            if "GEFS" in model:
                mask = mask_GEFS
            elif "IITM" in model:
                mask = mask_IITM
            elif "ECMWF" in model:
                mask = mask_ECMWF
            else:
                mask = None
        else:  # Common Period / MME / 2MME
            mask = mask_MME

        if mask is not None:
            all_rpss["unet"][period][model] = unet_da.where(~mask)

#use ELR ECMWF mask for ECMWF
all_rpss["ELR"]["Full period"]["ECMWF"] = all_rpss["ELR"]["Full period"]["ECMWF"].where(~mask_ECMWF)

#assign MME and 2MME to the common period
all_rpss["ELR"]["Common Period"]["3MME"] = all_rpss["ELR"]["MME"]["MME"]
all_rpss["ELR"]["Common Period"]["2MME"] = all_rpss["ELR"]["2MME"]["2MME"]
all_rpss["unet"]["Common Period"]["3MME"] = all_rpss["unet"]["MME"]["MME"]
all_rpss["unet"]["Common Period"]["2MME"] = all_rpss["unet"]["2MME"]["2MME"]

#mask RAW models for common period
all_rpss["raw"]["Common Period"]["ECMWF"] = all_rpss["raw"]["Common Period"]["ECMWF"].where(~mask_MME)
all_rpss["raw"]["Common Period"]["GEFS"] = all_rpss["raw"]["Common Period"]["GEFS"].where(~mask_MME)
all_rpss["raw"]["Common Period"]["IITM"] = all_rpss["raw"]["Common Period"]["IITM"].where(~mask_MME)


In [ ]:
def plot_rpss_map(da, ax, model_name, vmin=-0.2, vmax=0.2, cmap="bwr"):
    # Ensure latitude is ascending
    if "lat" in da.dims and da.lat[0] > da.lat[-1]:
        da = da.sortby("lat")

    im = da.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        add_colorbar=False
    )
    ax.coastlines()

    for shape_name in ['indian_borders']:
        reader = cartopy.io.shapereader.Reader(f'shapes/{shape_name}.shp')
        ax.add_geometries(reader.geometries(), ccrs.PlateCarree(), facecolor='none', edgecolor='black')

    # Compute mean, min, max
    mean_val = float(da.mean().values)
    min_val = float(da.min().values)
    max_val = float(da.max().values)

    # Subplot title above the map
    ax.set_title(f"{model_name}", fontsize=15, fontweight="bold", pad=20)

    # Gridlines
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.5, alpha=0.7)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

    # Mean/min/max below the map
    # ax.text(
    #     0.5, 1.05,
    #     f"Mean={mean_val:.2f} | Min={min_val:.2f} | Max={max_val:.2f}",
    #     transform=ax.transAxes,
    #     ha="center",
    #     fontsize=11,
    #     fontweight="normal"
    # )

    return im

def make_panel(arch, period, models, figpath):
    n = len(models)
    fig, axes = plt.subplots(
        1, n,
        figsize=(3.5*n, 4),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True
    )
    if n == 1:
        axes = [axes]

    im = None
    for ax, model in zip(axes, models):
        if model not in all_rpss[arch][period]:
            ax.set_visible(False)
            continue
        da = all_rpss[arch][period][model]
        im = plot_rpss_map(da, ax, model)

    # Shared horizontal colorbar
    if im is not None:
        cbar = fig.colorbar(im, ax=axes, orientation="horizontal", fraction=0.05, pad=0.08)
        cbar.set_label("RPSS", fontsize=15)


    plt.savefig(figpath, dpi=300, bbox_inches="tight", format="pdf")
    plt.show()
    plt.close(fig)

make_panel("ELR", "Full period", ["GEFS", "IITM", "ECMWF"], "figures/rpss/ELR_Full.pdf")
make_panel("unet", "Full period", ["GEFS", "IITM", "ECMWF"], "figures/rpss/UNet_Full.pdf")

# Common period + 2MME + MME
make_panel("ELR", "Common Period", ["GEFS", "IITM", "ECMWF", "2MME", "3MME"], "figures/rpss/ELR_Common.pdf")
make_panel("unet", "Common Period", ["GEFS", "IITM", "ECMWF", "2MME", "3MME"], "figures/rpss/UNet_Common.pdf")


# Train val test plot

In [ ]:
train_rpss = xr.open_dataarray('outputs/Full Period/IITM_IMD/unet_rpss_train_wk3-4.nc').mean('bootstrap').where(~mask_IITM)
val_rpss = xr.open_dataarray('outputs/Full Period/IITM_IMD/unet_rpss_val_wk3-4.nc').mean('bootstrap').where(~mask_IITM)
test_rpss = xr.open_dataarray('outputs/Full Period/IITM_IMD/unet_rpss_test_wk3-4.nc').mean('bootstrap').where(~mask_IITM)

#plot all three
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw={"projection": ccrs.PlateCarree()}, constrained_layout=True)

for ax, da, split in zip(axes, [train_rpss, val_rpss, test_rpss], ["Train", "Validation", "Test"]):
    im = plot_rpss_map(
        da,
        ax,
        model_name=f"{split} RPSS",
        vmin=-0.2,
        vmax=0.2,
        cmap="bwr"
    )
#add mean, min, max below each subplot
    mean_val = float(da.mean().values)
    min_val = float(da.min().values)
    max_val = float(da.max().values)
    # ax.text(
    #     0.5, 1.2,
    #     f"Mean={mean_val:.2f} | Min={min_val:.2f} | Max={max_val:.2f}",
    #     transform=ax.transAxes,
    #     ha="center",
    #     fontsize=15,
    #     fontweight="normal"
    #  )
# Shared colorbar
cbar = fig.colorbar(im, ax=axes, orientation="horizontal", fraction=0.05, pad=0.08)
cbar.set_label("RPSS", fontsize=15)
plt.savefig("figures/rpss/IITM_Full_Train_Val_Test.pdf", dpi=300, bbox_inches="tight")
plt.show()

